In [1]:
import pandas as pd
import pyarrow

url = "C:/Users/Michael/Downloads/popec22.parquet"

df = pd.read_parquet(url, engine='pyarrow')

excluir = ['P00','P01','P02','P03']

agrupacion = [c for c in df.columns if c not in excluir]

# Al obtener el número máximo del número de persona obtendremos la cantidad de personas del hogar
personasporhogar = (
    df.groupby(agrupacion)
      .agg(numpersonas=('P00','max'))
      .reset_index()
)

# Excluir no hogares (revisar explicación cap anterior)
personasporhogar = personasporhogar[personasporhogar['INH']>0]

In [2]:
import altair as alt

# Elección de todos los datos de su parroquia
# Provincia: Sucumbíos, Cantón: Sucumbíos, Parroquia: La Sofía
# código: 210552
parroquia = personasporhogar[
    (personasporhogar['I01'] == 21) & 
    (personasporhogar['I02'] == 5) & 
    (personasporhogar['I03'] == 52)
]

# Obtenemos la probabilidad por tamaño de hogar
freq_parroquia = (
    parroquia['numpersonas']
    .value_counts(normalize=True)
    .sort_index()
    .reset_index()
)

# Graficar resultado
chart = alt.Chart(freq_parroquia).mark_bar(
    size=25
).encode(
    x=alt.X(
        'numpersonas:O',
        sort=None,
        title='Número de personas en el hogar',
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        'proportion:Q',
        title='Porcentaje de hogares',
        axis=alt.Axis(format='.0%')
    ),
    tooltip=[
        alt.Tooltip('numpersonas:O', title='Número de miembros'), 	
        alt.Tooltip('proportion:Q', title='Probabilidad del evento', format='.2%')
    ]
).properties(
    title="Distribución del tamaño del hogar",
    width=700,
    height=400
)

chart

alt.Chart(...)

In [3]:
# Probabilidades de la población
freq_df = (
    personasporhogar['numpersonas']
    .value_counts(normalize=True)
    .sort_index()
    .reset_index()
)

# Unir distribuciones
freq = freq_df.rename(columns={'proportion': 'poblacion'}).merge(
    freq_parroquia.rename(columns={'proportion': 'muestra'}), 
    on='numpersonas', how='left'
)

# Cambiamos la estructura de la tabla
freq_resize = freq.melt(
    id_vars='numpersonas',
    var_name='grupo',
    value_name='porcentaje'
)

chart = alt.Chart(freq_resize).mark_bar(
    size=10 # más ancho (porque hay xOffset)
).encode(
    x=alt.X(
        'numpersonas:O',
        title='Número de personas en el hogar',
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        'porcentaje:Q',
        title='Porcentaje de hogares',
        axis=alt.Axis(format='.1%')
    ),
    color=alt.Color(
        'grupo:N',
        sort=['poblacion','muestra'],
        title='',
        legend=alt.Legend(orient='top'),   # mueve la leyenda arriba
        scale=alt.Scale(
            domain=['poblacion','muestra'],
            range=['#013440', '#593954']
        )
    ),
    xOffset=alt.XOffset(
        'grupo:N',
        sort=['poblacion','muestra']   # importante también aquí
    ),
    tooltip=[
        alt.Tooltip('numpersonas:O', title='Personas'),
        alt.Tooltip('grupo:N', title='Grupo'),
        alt.Tooltip('porcentaje:Q', title='Porcentaje',format='.2%')
    ]
).properties(
    title="Distribución del tamaño del hogar",
    width=700,
    height=400
)

chart

alt.Chart(...)

In [4]:
# Función que toma una selección aleatoria según tamaño
sizepop = len(personasporhogar)
def comparacion_muestrales(tamano_muestra=100):

    if tamano_muestra > sizepop:
        print(f"El tamaño de la muestra ({tamano_muestra}) es mayor que la población ({sizepop})")
        return

    muestra = personasporhogar.sample(
        n=min(tamano_muestra, sizepop),   # ← protección extra
        random_state=123
    )

    # Obtenemos la probabilidad por tamaño de hogar
    freq_muestra = (
        muestra['numpersonas']
        .value_counts(normalize=True)
        .sort_index()
        .reset_index()
    )

    # Unir distribuciones muestra - población
    freq = freq_df.rename(columns={'proportion': 'poblacion'}).merge(
        freq_muestra.rename(columns={'proportion': 'muestra'}), 
        on='numpersonas', how='left'
    )

    # Cambiamos la estructura de la tabla
    freq_resize = freq.melt(
        id_vars='numpersonas',
        var_name='grupo',
        value_name='porcentaje'
    )

    grafico = alt.Chart(freq_resize).mark_bar(
        size=10 # más ancho (porque hay xOffset)
    ).encode(
        x=alt.X(
            'numpersonas:O',
            title='Número de personas en el hogar',
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y(
            'porcentaje:Q',
            title='Porcentaje de hogares',
            axis=alt.Axis(format='.1%')
        ),
        color=alt.Color(
            'grupo:N',
            sort=['poblacion','muestra'],
            title='',
            legend=alt.Legend(orient='top'),   # mueve la leyenda arriba
            scale=alt.Scale(
                domain=['poblacion','muestra'],
                range=['#013440', '#593954']
            )
        ),
        xOffset=alt.XOffset(
            'grupo:N',
            sort=['poblacion','muestra']   # importante también aquí
        ),
        tooltip=[
            alt.Tooltip('numpersonas:O', title='Personas'),
            alt.Tooltip('grupo:N', title='Grupo'),
            alt.Tooltip('porcentaje:Q', title='Porcentaje',format='.2%')
        ]
    ).properties(
        title="Distribución del tamaño del hogar",
        width=700,
        height=400
    )

    return grafico

comparacion_muestrales(tamano_muestra=100)

alt.Chart(...)

In [5]:
print(f'El número de hogares en la parroquia Sofía es {parroquia.shape[0]}')

El número de hogares en la parroquia Sofía es 19


In [6]:
import ipywidgets as widgets
from ipywidgets import interact

step_size = 1000

interact(
    comparacion_muestrales,
    tamano_muestra=widgets.IntSlider(
        min=100,
        max=sizepop,
        step=100,
        value=500,
        description='n',
        continuous_update=False
    )
)

interactive(children=(IntSlider(value=500, continuous_update=False, description='n', max=2780237, min=100, ste…

<function __main__.comparacion_muestrales(tamano_muestra=100)>